### **Import modules and setting the env vars**

In [1]:
import imaplib, ssl, email, os, json, re
from email import policy
from pathlib import Path
from dotenv import load_dotenv

from bs4 import BeautifulSoup
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_groq import ChatGroq

import sqlite3

In [2]:
env_path = Path.cwd().parent / "private" / ".env"
load_dotenv(env_path)

imap_server = os.environ["IITB_IMAP_SERVER"]
email_address = os.environ["IITB_EMAIL"]
access_token = os.environ["IITB_IMAP_ACCESS_TOKEN"]

In [3]:
# 1. Create a modern default SSL context
context = ssl.create_default_context()

# 2. Lower the restriction to allow OpenSSL's standard 'DEFAULT' ciphers
# This bridges the gap with servers requiring legacy key exchanges.
# IITB IMAP server need this
context.set_ciphers('DEFAULT')

context

In [4]:
try:
    print(f"Connecting securely to {imap_server}...")

    # 3. Pass the custom context into IMAP4_SSL
    mail = imaplib.IMAP4_SSL(imap_server, 993, ssl_context=context)
    mail.login(email_address, access_token)
    print("Login successful!")
    mail.logout()

except ssl.SSLError as ssl_err:
    print(f"SSL Specific Error: {ssl_err}")
except Exception as e:
    print(f"Connection failed: {e}")

Connecting securely to imap.iitb.ac.in...
Login successful!


---

### **Now We extract event details in json format using free langchain api**

In [5]:
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

AUDIENCES = ["Public", "Institute-wide", "Department-specific", "Club-specific", "Personal"]

SYSTEM_PROMPT = f"""Extract title, venue, date, time, organizer and audience from the email below.
Audience must be exactly one of: {", ".join(AUDIENCES)}. If a field is not in the email, use null. 
Also give the date and time in proper format. In some mail it could be possible date and time are given together, you have to extract it.
Return only valid JSON with the keys: title, venue, date, time, organizer, audience."""

In [6]:
def get_body(msg):
    part = msg.get_body(preferencelist=("plain", "html"))
    if part is None:
        return ""
    content = part.get_content()
    if part.get_content_type() == "text/html":
        content = BeautifulSoup(content, "html.parser").get_text(separator="\n")
    return content.strip()

def parse_json(text):
    """Pull the first {...} block out of the model reply and validate it."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        data = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    keys = ["title", "venue", "date", "time", "organizer", "audience"]
    data = {k: data.get(k) for k in keys}
    if data["audience"] not in AUDIENCES:
        data["audience"] = None
    return data

def extract_event(subject, sender, date, body):
    text = f"Subject: {subject}\nFrom: {sender}\nDate: {date}\n\n{body[:3000]}"  # TinyLlama context is small
    reply = model.invoke([SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=text)])
    parsed = parse_json(reply.content)
    return parsed, reply.content

In [7]:
events = []

try:
    mail = imaplib.IMAP4_SSL(imap_server, 993, ssl_context=context)
    mail.login(email_address, access_token)

    mail.select("INBOX", readonly=True)
    status, messages = mail.search(None, "ALL")

    for num in messages[0].split()[::-1][:10]:      # latest 10 while testing
        _, data = mail.fetch(num, "(BODY.PEEK[])")
        msg = email.message_from_bytes(data[0][1], policy=policy.default)

        parsed, raw = extract_event(msg["Subject"], msg["From"], msg["Date"], get_body(msg))

        if parsed:
            print(json.dumps(parsed, indent=2))
            events.append(parsed)
        else:
            print(json.dumps({"error": "could not parse model output"}))

    mail.logout()

except ssl.SSLError as ssl_err:
    print(f"SSL Specific Error: {ssl_err}")
except Exception as e:
    print(f"Connection failed: {e}")

{
  "title": "Team Endeavour Fresher' Orientation",
  "venue": null,
  "date": "2026-10-15",
  "time": "18:30:00",
  "organizer": "General Secretary Technical Affairs",
  "audience": "Club-specific"
}
{
  "title": "Figma 101: A Beginner Friendly Workshop",
  "venue": null,
  "date": "2026-10-14",
  "time": null,
  "organizer": "GS Cultural Affairs IIT Bombay",
  "audience": "Institute-wide"
}
{
  "title": "HEISENBERG",
  "venue": "LH101",
  "date": "2026-10-17",
  "time": null,
  "organizer": "General Secretary Technical Affairs",
  "audience": "Institute-wide"
}
{
  "title": "Dandiya Night \u201926",
  "venue": null,
  "date": "2026-10-16",
  "time": null,
  "organizer": "GS Cultural Affairs IIT Bombay",
  "audience": "Institute-wide"
}
{
  "title": "Dandiya Night \u201926",
  "venue": null,
  "date": "2026-10-16",
  "time": null,
  "organizer": "GS Cultural Affairs IIT Bombay",
  "audience": "Institute-wide"
}
{
  "title": "Concert in the memory of Prof. Pushpak Bhattacharyya",
  "ve

---

### **Now We save the event details in raw sql database**

In [8]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)    # create 'data/' if it doesn't exist

# Save events to raw sql database
EVENTS_DB = DATA_DIR / "events.db"

# Open (or create) the database file
conn = sqlite3.connect(EVENTS_DB)
cursor = conn.cursor()

# Create events table
cursor.execute(f'''
CREATE TABLE IF NOT EXISTS events (
    id        INTEGER PRIMARY KEY AUTOINCREMENT,
    title     TEXT NOT NULL,
    venue     TEXT,
    date      DATE,
    time      TIME,
    organizer TEXT,
    audience  TEXT
)
''')
conn.commit()

print("Events database path:", os.path.relpath(EVENTS_DB))

Events database path: ../data/events.db


In [9]:
# Insert extracted events if the table is empty
count = cursor.execute("SELECT COUNT(*) FROM events").fetchone()[0]

if count == 0:
    cursor.executemany(
        """INSERT INTO events (title, venue, date, time, organizer, audience)
        VALUES (?, ?, ?, ?, ?, ?)""",
        [
            (
                event.get("title"),
                event.get("venue"),
                event.get("date"),
                event.get("time"),
                event.get("organizer"),
                event.get("audience")
            )
            for event in events
        ]
    )
    conn.commit()
    print(f"{len(events)} events added")
else:
    print(f"Table already has {count} rows")

conn.close()

10 events added


---